# E007 - Optimisation contextuelle scannability-first

Ce notebook optimise conjointement les paramètres Stage-1, SRPG et robustesse. La scannabilité exacte reste une contrainte dure. CLIP-aesthetic et CLIPScore ne départagent que les résultats lisibles. Le plan factoriel sépare prompt, seed et payload, puis un mini-modèle apprend à recommander des paramètres pour une nouvelle demande.

In [ ]:
EXPERIMENT_NAME = "e007-contextual-v1"
SEARCH_TRIALS = 72
CALIBRATION_CONFIGS = 8
TOP_CONFIGS = 5
ADAPTIVE_POOL = 1024
ADAPTIVE_ATTEMPTS = 6
STRICT_CANDIDATES_TARGET = 3
USER_PROMPT = "premium botanical wine label, elegant flowers and leaves, detailed engraving"
USER_PAYLOAD = "https://example.prooftag.test/t/e007-user"
USER_SEED = 4242

## 1. Porte GPU exclusive
Cette cellule doit être exécutée avant tout import CUDA. Elle arrête la campagne si vLLM, l'API QR ou un ancien kernel consomme encore la RTX.

In [ ]:
from prooftag_qr.optimization import require_exclusive_gpu

require_exclusive_gpu()
print("GPU libre avant chargement : OK")

## 2. Chargement diffusion, CLIP et stockage reprenable
CLIP et le prédicteur esthétique restent sur CPU pour réserver les 20 Gio GPU à SRPG. Le premier chargement télécharge les poids dans le PVC de cache.

In [ ]:
import csv
import json
import shutil
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import optuna
import torch
from IPython.display import Markdown, display

from prooftag_qr.advisor import ContextualParameterAdvisor
from prooftag_qr.config import Settings
from prooftag_qr.experiments import (
    SRPGTrial,
    aggregate_confirmation,
    sample_e007_trial,
    select_delivery_candidate,
    trial_rank_key,
)
from prooftag_qr.optimization import (
    E007Experiment,
    ExperimentContext,
    factorial_contexts,
    holdout_contexts,
)

settings = Settings(
    data_dir=Path("/data"),
    model_cache_dir=Path("/cache"),
    default_backend="controlnet",
    controlnet_pipeline_mode="img2img",
    device="cuda",
    srpg_enabled=False,
    guided_rediffusion_enabled=False,
    latent_refinement_enabled=False,
)
experiment = E007Experiment(settings, EXPERIMENT_NAME)
display(
    Markdown(
        f"**GPU :** `{torch.cuda.get_device_name(0)}`  \
**Dossier :** `{experiment.run_dir}`"
    )
)

## 3. Plan factoriel sans confusion
L'axe prompt conserve payload et seed. L'axe seed conserve prompt et payload. L'axe payload conserve prompt et seed. Ainsi, E007 peut enfin mesurer séparément chaque source de variabilité.

In [ ]:
contexts = list(factorial_contexts())
for context in contexts:
    print(f"{context.context_id:22s} axis={context.axis:7s} seed={context.seed}")
print(f"{len(contexts)} contextes appariés")

## 4. Recherche TPE sur toutes les dimensions utiles
Les dimensions continues rendent un produit cartésien infini. TPE explore simultanément Stage-1, Stage-2, seuils SRL, robustesse différentiable, seed, eta et profil de negative prompt. Chaque essai est persisté ; relancer la cellule reprend l'étude.

In [ ]:
storage = f"sqlite:///{experiment.run_dir / 'optuna.db'}"
sampler = optuna.samplers.TPESampler(
    seed=20260721,
    multivariate=True,
    group=True,
    n_startup_trials=24,
)
study = optuna.create_study(
    study_name=EXPERIMENT_NAME,
    storage=storage,
    load_if_exists=True,
    sampler=sampler,
    directions=["maximize", "maximize", "maximize", "minimize"],
)


def objective(optuna_trial):
    context = contexts[optuna_trial.number % len(contexts)]
    config = sample_e007_trial(optuna_trial, name=f"tpe-{optuna_trial.number:04d}")
    row = experiment.execute("search", context, config)
    if row["status"] != "ok":
        raise optuna.TrialPruned(row.get("error", "trial error"))
    optuna_trial.set_user_attr("context_id", context.context_id)
    optuna_trial.set_user_attr("strict_all", row["strict_all"])
    optuna_trial.set_user_attr("result_key", row["key"])
    scan_priority = row["pass_rate"] + 2.0 * float(row["strict_all"])
    print(
        f"{row['key']} scan={row['passed']}/{row['validations']} "
        f"aes={row['clip_aesthetic']:.3f} clip={row['clip_score']:.3f}"
    )
    return scan_priority, row["clip_aesthetic"], row["clip_score"], row["duration_seconds"]


completed = sum(
    trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials
)
failed = sum(trial.state == optuna.trial.TrialState.FAIL for trial in study.trials)
pruned = sum(trial.state == optuna.trial.TrialState.PRUNED for trial in study.trials)
print(f"Reprise : {completed} complets, {failed} interrompus/échoués, {pruned} élagués")
remaining = max(0, SEARCH_TRIALS - completed)
if remaining:
    study.optimize(objective, n_trials=remaining, gc_after_trial=True)
completed = sum(
    trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials
)
print(f"Étude : {completed}/{SEARCH_TRIALS} essais complets")

## 5. Résultats et importance de tous les paramètres
Le graphique ne transforme jamais esthétique et scan en moyenne. Les points stricts 26/26 sont identifiés séparément.

In [ ]:
from optuna.importance import FanovaImportanceEvaluator, get_param_importances

search_rows = [
    row for row in experiment.rows() if row.get("phase") == "search" and row.get("status") == "ok"
]
if not search_rows:
    raise RuntimeError("Aucun résultat search/ok : terminer d'abord la recherche Optuna")
ranked_search = sorted(search_rows, key=trial_rank_key)
print(
    f"{len(search_rows)} résultats. Top recherche "
    "(scan strict, puis CLIP-aesthetic et CLIPScore)",
    flush=True,
)
for row in ranked_search[:10]:
    print(
        f"{row['trial']:10s} {row['context_id']:22s} scan={row['passed']:2d}/26 "
        f"aes={row['clip_aesthetic']:.3f} clip={row['clip_score']:.3f}"
    )

fig, axis = plt.subplots(figsize=(10, 6))
colors = [row["clip_score"] for row in search_rows]
markers = [120 if row["strict_all"] else 35 for row in search_rows]
scatter = axis.scatter(
    [row["pass_rate"] for row in search_rows],
    [row["clip_aesthetic"] for row in search_rows],
    c=colors,
    s=markers,
    cmap="viridis",
    alpha=0.8,
)
axis.set_xlabel("Taux de validation QR")
axis.set_ylabel("CLIP-aesthetic")
axis.axvline(1.0, color="red", linestyle="--", label="porte 26/26")
axis.grid(alpha=0.25)
axis.legend()
fig.colorbar(scatter, label="CLIPScore")
fig.tight_layout()
fig.savefig(experiment.run_dir / "search-objectives.png", dpi=150)
display(fig)
plt.close(fig)

print("Calcul fANOVA borné en cours...", flush=True)
importance = get_param_importances(
    study,
    evaluator=FanovaImportanceEvaluator(n_trees=32, max_depth=16, seed=20260722),
    target=lambda trial: float(trial.values[0]),
)
print(f"fANOVA terminée : {len(importance)} paramètres classés", flush=True)
(experiment.run_dir / "parameter-importance.json").write_text(
    json.dumps(importance, indent=2), encoding="utf-8"
)
top_importance = list(importance.items())[:15]
fig, axis = plt.subplots(figsize=(10, 7))
axis.barh(
    [name for name, _ in reversed(top_importance)],
    [value for _, value in reversed(top_importance)],
)
axis.set_xlabel("Importance pour la scannabilité")
axis.set_title("Top 15 paramètres TPE")
axis.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(experiment.run_dir / "parameter-importance.png", dpi=150)
display(fig)
plt.close(fig)
importance

## 6. Promotion factorielle puis confirmation sur prompts jamais vus
Les huit meilleures configurations du screening sont d'abord rejouées sur les douze contextes factoriels. Les cinq plus robustes sont ensuite confirmées sur quatre contextes holdout. Une configuration incomplète, en erreur ou sous 26/26 n'est jamais présentée comme stricte.

In [ ]:
screening_rows = ranked_search[:CALIBRATION_CONFIGS]
screening_configs = [SRPGTrial(**row["parameters"]) for row in screening_rows]
screening_names = {config.name for config in screening_configs}
calibration_rows = [
    row
    for row in experiment.rows()
    if row.get("phase") == "calibration"
    and row.get("status") == "ok"
    and row.get("trial") in screening_names
]
calibration_done = {(row["trial"], row["context_id"]) for row in calibration_rows}
calibration_pending = [
    (context, config)
    for context in contexts
    for config in screening_configs
    if (config.name, context.context_id) not in calibration_done
]
calibration_total = len(contexts) * len(screening_configs)
print(
    f"Calibration : {len(calibration_done)}/{calibration_total} terminées, "
    f"{len(calibration_pending)} restantes",
    flush=True,
)
for index, (context, config) in enumerate(calibration_pending, start=1):
    position = len(calibration_done) + index
    print(
        f"[calibration {position}/{calibration_total}] START "
        f"{config.name} × {context.context_id}",
        flush=True,
    )
    row = experiment.execute("calibration", context, config)
    print(
        f"[calibration {position}/{calibration_total}] {row['status']} "
        f"scan={row.get('passed', 0)}/{row.get('validations', 26)} "
        f"durée={row.get('duration_seconds', 0):.1f}s",
        flush=True,
    )
calibration_rows = [
    row
    for row in experiment.rows()
    if row.get("phase") == "calibration"
    and row.get("status") == "ok"
    and row.get("trial") in screening_names
]
if len(calibration_rows) != calibration_total:
    raise RuntimeError(
        f"Calibration incomplète : {len(calibration_rows)}/{calibration_total} résultats ok"
    )
calibration_aggregates = aggregate_confirmation(
    calibration_rows, expected_cases=len(contexts)
)
config_by_name = {config.name: config for config in screening_configs}
top_names = [row["trial"] for row in calibration_aggregates[:TOP_CONFIGS]]
top_configs = [config_by_name[name] for name in top_names]
(experiment.run_dir / "calibration-aggregates.json").write_text(
    json.dumps(calibration_aggregates, indent=2), encoding="utf-8"
)
print("Configurations promues après les 12 contextes factoriels :")
for row in calibration_aggregates[:TOP_CONFIGS]:
    print(row)
lookup = {(row["trial"], row["context_id"]): row for row in calibration_rows}
heatmap = [
    [lookup[(config.name, context.context_id)]["pass_rate"] for context in contexts]
    for config in screening_configs
]
fig, axis = plt.subplots(figsize=(14, 6))
image = axis.imshow(heatmap, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
axis.set_xticks(range(len(contexts)), [item.context_id for item in contexts], rotation=55)
axis.set_yticks(
    range(len(screening_configs)), [item.name for item in screening_configs]
)
axis.set_title("Calibration : taux de lecture par configuration et contexte")
fig.colorbar(image, label="Taux de validation QR")
fig.tight_layout()
fig.savefig(experiment.run_dir / "calibration-heatmap.png", dpi=150)
display(fig)
plt.close(fig)

holdouts = list(holdout_contexts())
top_name_set = {config.name for config in top_configs}
holdout_rows = [
    row
    for row in experiment.rows()
    if row.get("phase") == "holdout"
    and row.get("status") == "ok"
    and row.get("trial") in top_name_set
]
holdout_done = {(row["trial"], row["context_id"]) for row in holdout_rows}
holdout_pending = [
    (context, config)
    for context in holdouts
    for config in top_configs
    if (config.name, context.context_id) not in holdout_done
]
holdout_total = len(holdouts) * len(top_configs)
print(
    f"Holdouts : {len(holdout_done)}/{holdout_total} terminés, "
    f"{len(holdout_pending)} restants",
    flush=True,
)
for index, (context, config) in enumerate(holdout_pending, start=1):
    position = len(holdout_done) + index
    print(
        f"[holdout {position}/{holdout_total}] START "
        f"{config.name} × {context.context_id}",
        flush=True,
    )
    row = experiment.execute("holdout", context, config)
    print(
        f"[holdout {position}/{holdout_total}] {row['status']} "
        f"scan={row.get('passed', 0)}/{row.get('validations', 26)} "
        f"durée={row.get('duration_seconds', 0):.1f}s",
        flush=True,
    )
holdout_rows = [
    row
    for row in experiment.rows()
    if row.get("phase") == "holdout"
    and row.get("status") == "ok"
    and row.get("trial") in top_name_set
]
if len(holdout_rows) != holdout_total:
    raise RuntimeError(
        f"Holdouts incomplets : {len(holdout_rows)}/{holdout_total} résultats ok"
    )
aggregates = aggregate_confirmation(holdout_rows, expected_cases=len(holdouts))
for row in aggregates:
    print(row)
(experiment.run_dir / "holdout-aggregates.json").write_text(
    json.dumps(aggregates, indent=2), encoding="utf-8"
)

## 7. Mini-modèle adaptatif
ExtraTrees apprend les interactions non linéaires entre prompt projeté par CLIP, structure du brut, QR et paramètres. La validation croisée sépare les contextes afin de mesurer la généralisation à une demande jamais vue.

In [ ]:
training_rows = [
    row
    for row in experiment.rows()
    if row.get("phase") in {"search", "calibration", "holdout"}
]
advisor = ContextualParameterAdvisor(trees=384)
advisor_report = advisor.fit(training_rows)
advisor.save(experiment.run_dir / "contextual-parameter-advisor.joblib")
(experiment.run_dir / "advisor-report.json").write_text(
    json.dumps(advisor_report, indent=2), encoding="utf-8"
)
advisor_report

## 8. Simulation d'une nouvelle demande utilisateur
Une première configuration globale produit le brut. Le mini-modèle classe ensuite 1024 variantes Stage-2 pour ce contexte précis. Jusqu'à six sont réellement générées ; la meilleure des sorties 26/26 est choisie sur CLIP-aesthetic puis CLIPScore. Sans sortie stricte, aucune image n'est livrée.

In [ ]:
user_context = ExperimentContext("adaptive-user", "adaptive", USER_PROMPT, USER_PAYLOAD, USER_SEED)
seed_config = top_configs[0]
seed_row = experiment.execute("adaptive", user_context, replace(seed_config, name="seed-global"))
if seed_row["status"] != "ok":
    raise RuntimeError(seed_row.get("error", "adaptive seed failed"))

pool_study = optuna.create_study(sampler=optuna.samplers.RandomSampler(seed=20260722))
candidate_pool = []
for index in range(ADAPTIVE_POOL):
    asked = pool_study.ask()
    sampled = sample_e007_trial(asked, name=f"advisor-{index:04d}")
    candidate_pool.append(
        replace(
            sampled,
            base_steps=seed_config.base_steps,
            base_strength=seed_config.base_strength,
            base_guidance_scale=seed_config.base_guidance_scale,
            base_controlnet_scale=seed_config.base_controlnet_scale,
            negative_prompt_profile=seed_config.negative_prompt_profile,
        )
    )
    pool_study.tell(asked, state=optuna.trial.TrialState.PRUNED)
recommendations = advisor.recommend(
    seed_row["context_features"], candidate_pool, limit=ADAPTIVE_ATTEMPTS
)
recommendation_report = [
    {
        "trial": item.trial.name,
        "parameters": asdict(item.trial),
        "predicted_pass_rate": item.predicted_pass_rate,
        "pass_rate_uncertainty": item.pass_rate_uncertainty,
        "predicted_clip_aesthetic": item.predicted_clip_aesthetic,
        "predicted_clip_score": item.predicted_clip_score,
    }
    for item in recommendations
]
(experiment.run_dir / "advisor-recommendations.json").write_text(
    json.dumps(recommendation_report, indent=2), encoding="utf-8"
)
adaptive_rows = [seed_row]
strict_count = int(seed_row["strict_all"])
for recommendation in recommendations:
    print(
        f"try {recommendation.trial.name}: predicted scan="
        f"{recommendation.predicted_pass_rate:.1%} ± {recommendation.pass_rate_uncertainty:.1%}"
    )
    row = experiment.execute("adaptive", user_context, recommendation.trial)
    adaptive_rows.append(row)
    strict_count += int(row.get("strict_all", False))
    if strict_count >= STRICT_CANDIDATES_TARGET:
        break
executed_predictions = {item.trial.name: item for item in recommendations}
executed_rows = [row for row in adaptive_rows[1:] if row.get("status") == "ok"]
if executed_rows:
    predicted = [executed_predictions[row["trial"]].predicted_pass_rate for row in executed_rows]
    observed = [row["pass_rate"] for row in executed_rows]
    fig, axis = plt.subplots(figsize=(6, 6))
    axis.scatter(predicted, observed, s=90)
    axis.plot([0, 1], [0, 1], linestyle="--", color="black")
    axis.set(xlim=(0, 1), ylim=(0, 1), xlabel="Scan prédit", ylabel="Scan observé")
    axis.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(experiment.run_dir / "advisor-predicted-vs-observed.png", dpi=150)
    display(fig)
selected = select_delivery_candidate(adaptive_rows)
if selected is None:
    display(Markdown("## REJET : aucune image 26/26, rien ne doit être livré."))
else:
    shutil.copy2(selected["image"], experiment.run_dir / "DELIVERY.png")
    (experiment.run_dir / "delivery.json").write_text(
        json.dumps(selected, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    display(
        Markdown(
            f"## LIVRABLE 26/26 : `{selected['trial']}` - "
            f"aesthetic={selected['clip_aesthetic']:.3f}, CLIPScore={selected['clip_score']:.3f}"
        )
    )

## 9. Export, validation physique et archive
Le CSV physique ne contient que les candidats stricts. L'archive finale permet une autopsie complète.

In [ ]:
all_rows = experiment.rows()
flat_rows = []
for row in all_rows:
    flat = {
        key: value
        for key, value in row.items()
        if not isinstance(value, (dict, list, tuple))
    }
    flat.update({f"parameter_{key}": value for key, value in row.get("parameters", {}).items()})
    flat.update(
        {f"context_{key}": value for key, value in row.get("context_features", {}).items()}
    )
    flat_rows.append(flat)
summary_fields = sorted({key for row in flat_rows for key in row})
with (experiment.run_dir / "run-summary.csv").open(
    "w", newline="", encoding="utf-8"
) as stream:
    writer = csv.DictWriter(stream, fieldnames=summary_fields)
    writer.writeheader()
    writer.writerows(flat_rows)
phase_summary = {}
for phase in sorted({row.get("phase", "unknown") for row in all_rows}):
    phase_rows = [row for row in all_rows if row.get("phase", "unknown") == phase]
    phase_summary[phase] = {
        "runs": len(phase_rows),
        "errors": sum(row.get("status") != "ok" for row in phase_rows),
        "strict": sum(bool(row.get("strict_all")) for row in phase_rows),
    }
(experiment.run_dir / "campaign-summary.json").write_text(
    json.dumps(phase_summary, indent=2), encoding="utf-8"
)
strict_rows = [
    row for row in all_rows if row.get("status") == "ok" and row.get("strict_all")
]
phone_path = experiment.run_dir / "phone-validation.csv"
with phone_path.open("w", newline="", encoding="utf-8") as stream:
    fields = [
        "trial",
        "context_id",
        "image",
        "device",
        "protocol",
        "attempts",
        "successes",
        "exact_payload",
        "notes",
    ]
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    for row in strict_rows:
        protocols = (
            "screen-front-30cm",
            "screen-angle-30deg",
            "screen-low-light",
            "print-5cm",
        )
        for protocol in protocols:
            writer.writerow(
                {
                    "trial": row["trial"],
                    "context_id": row["context_id"],
                    "image": row["image"],
                    "device": "",
                    "protocol": protocol,
                    "attempts": 10,
                    "successes": "",
                    "exact_payload": "",
                    "notes": "",
                }
            )
archive = shutil.make_archive(
    str(Path("/workspace/results") / EXPERIMENT_NAME),
    "gztar",
    root_dir=experiment.run_dir.parent,
    base_dir=EXPERIMENT_NAME,
)
print(f"Archive : {archive}")